<div style="width:100%; background-color:#181818; color:#f1f1f1; padding:30px 0; text-align:center; border-radius:10px;">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExODU4Zjk2NzZucnlsNDNjaWYzc2NwdXcxeG5tZThhOHV4MzZmYW5qOCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/eaBro4mE3Oe2I/giphy.gif" alt="Reinforced concrete design animation" width="300" style="max-width:100%; border-radius:10px;">
  <h3 style="color:#ffffff; margin:0 0 8px 0;"><b>Slabs, Direct Design Method</b></h3>
  <p style="margin:4px 0;"><b>Authors:</b> Msc. Ing. Carlos Andr&eacute;s Celi S&aacute;nchez &middot; Nicol&aacute;s Mora Bowen &middot; Juan Sebasti&aacute;n Baquero</p>
  <p style="margin:4px 0;"><b>Course:</b> Reinforced Concrete Design II</p>
</div>

### Libraries

In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Data

In [ ]:
fc = 280
fy = 4200
lx = 500
ly = 500
bw = 100
dvarF = 1.2
Ecb = 200000
Ecs = 200000

panel = 1                                       # 1 interior, 2 border or corner

condition1 = True                               # There must be a minimum of three continuous spans in each direction
condition2 = max(lx, ly) / min(lx, ly) <= 2     # Panels must be rectangular, with the ratio of the longer to the shorter span (center-to-center of supports) not exceeding 2.
condition3 = True                               # Successive span lengths, center-to-center of supports, in each direction must not differ by more than one-third of the longer span.
condition4 = True                               # Columns may be offset from either axis between centerlines of successive columns by no more than 10 percent of the span (in the direction of the offset).
condition5 = True                               # All loads must be due to gravity only, uniformly distributed over an entire panel, and the unfactored live load must not exceed twice the unfactored dead load.
condition6 = True                               # For a panel with beams between supports on all sides, the relative stiffness of the beams in the two perpendicular directions must satisfy

bv = 30
bh = 20


fi = 0.9
fiC = 0.75
rec= 2.5
Vu = 7610
tc = 40
tl = 40
alfas = 40

In [68]:
# 8.3.1.1. Two way slabs without interior beams
H = np.ceil(np.max([lx, ly]) / 33 / 5) * 5

#### Condition 6

In [69]:
ala_beam = min(bh - H, 4*H)

match panel:
    case 1:
        #------------ Iner Beam ------------#
        ala_b = bv + 2*ala_beam
        ala_h = H
        area_ala = ala_b * ala_h
        cg_ala = H/2
        I_ala = ala_b*ala_h**(3)/12

        alma_b = bv
        alma_h = bh - H
        area_alma = alma_b * alma_h
        cg_alma = H + alma_h/2
        I_alma = alma_b*alma_h**(3)/12

        areas = np.array([area_ala, area_alma])
        iners = np.array([I_ala, I_alma])
        yi    = np.array([cg_ala, cg_alma])
        a_y   = areas * yi


        ycg = np.sum(a_y) / np.sum(areas)
        
        Ibi = []
        for i in np.arange(0,len(areas)):
            Ibi.append(iners[i] + areas[i]*(yi[i] - ycg)**(2))

        Ib = np.sum(Ibi)

        #------------ Iner Slab ------------#
        L_colb1 = 2 * ly / 2                    # beams spanning in x: half panel on each side, measured in y
        L_colb2 = 2 * lx / 2                    # beams spanning in y: half panel on each side, measured in x
        Is1 = L_colb1 * H**(3) /12
        Is2 = L_colb2 * H**(3) /12

        #------------ αf ------------#     
        alfa_f1 = (Ecb*Ib) / (Ecs*Is1)
        alfa_f2 = (Ecb*Ib) / (Ecs*Is2)

        #------------ Condition 6 (8.10.2.7) ------------#
        ratio6 = alfa_f1*ly**(2) / (alfa_f2*lx**(2))
        condition6 = 0.2 <= ratio6 <= 5.0

        #------------ H min, Table 8.3.1.2 (mks-metric, ACI 318 Appendix C) ------------#
        alfa_fm = (2*alfa_f1 + 2*alfa_f2) / 4   # interior panel: four T-beams
        ln = np.max([lx, ly]) - bv              # clear span in the long direction, face-to-face of beams
        beta = (np.max([lx, ly]) - bv) / (np.min([lx, ly]) - bv)

        if alfa_fm <= 0.2:
            Hmin = np.max([lx, ly]) / 33        # 8.3.1.1 applies
        elif alfa_fm <= 2.0:
            Hmin = max(ln*(0.8 + fy/14000) / (36 + 5*beta*(alfa_fm - 0.2)), 12.5)
        else:
            Hmin = max(ln*(0.8 + fy/14000) / (36 + 9*beta), 9)

        conditions = [condition1, condition2, condition3, condition4, condition5, condition6]
        wy = (lx/ly)**(4)
        porc = wy / (np.ceil(wy) + 1) * 100

        if porc > 75:
            mensaje = 'one way slab'
        elif all(conditions):
            mensaje = 'two way slab - Direct Design Method applies'
            
            print('='*120)
            print(f'Ib = {Ib:,.2f} cm4 (T-beam)')
            print(f'ycg = {ycg:.2f} cm, measured from the top of slab')
            print(f'Is1 = {Is1:,.2f} cm4, Is2 = {Is2:,.2f} cm4')
            print(f'αf1 = {alfa_f1:.2f}, αf2 = {alfa_f2:.2f}, αfm = {alfa_fm:.2f}')
            print(f'Condition 6: 0.2 <= {ratio6:.2f} <= 5.0 -> {condition6}')
            print(f'ln = {ln:.2f} cm, β = {beta:.2f}')
            print(f'H min (8.3.1.2) = {Hmin:.2f} cm, H used = {H:.2f} cm -> {"OK" if H >= Hmin else "INCREASE H"}')
            print('='*120)
            
        else:
            failed = [i + 1 for i, c in enumerate(conditions) if not c]
            mensaje = f'two way slab - all DDM conditions must be satisfied (fails in condition: {failed})'

        print('='*120)
        print(f'Slab behavior = {mensaje}')
        print(f'H min = {H:.2f} cm')
        print(f'%  = {porc:.2f}')
        print('='*120)


        
    case 2:
        pass
    case _:
        raise ValueError("none")

Slab behavior = one way slab
H min = 30.00 cm
%  = 87.48


C:\Users\norma\AppData\Local\Temp\ipykernel_17100\2592158548.py:24: RuntimeWarning: divide by zero encountered in scalar divide
  ycg = np.sum(a_y) / np.sum(areas)
c:\Users\norma\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


### One-Way and Two-Way Action

In [70]:


wy = (lx/ly)**(4)
porc = wy / (np.ceil(wy) + 1) * 100

if porc > 75:
    mensaje = 'one way slab'
elif all(conditions):
    mensaje = 'two way slab - Direct Design Method applies'
else:
    failed = [i + 1 for i, c in enumerate(conditions) if not c]
    mensaje = f'two way slab - all DDM conditions must be satisfied (fails in condition: {failed})'

print('='*120)
print(f'Slab behavior = {mensaje}')
print(f'H min = {H:.2f} cm')
print(f'%  = {porc:.2f}')
print('='*120)

Slab behavior = one way slab
H min = 30.00 cm
%  = 87.48
